# GPAGD Demo: Poisson 1D
This notebook demonstrates GPAGD on the Poisson 1D benchmark.

In [ ]:
!pip install git+https://github.com/mohsenmostafa/GPAGD-Optimizer.git

In [ ]:
import torch
import torch.nn as nn
from gpagd.optimizers import GeometricPhysicsGD
from gpagd.utils import PCAManifoldProjector, local_entropy_1d
from benchmarks.poisson import Poisson1D
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
problem = Poisson1D()
model = nn.Sequential(nn.Linear(1,50), nn.Tanh(), nn.Linear(50,50), nn.Tanh(), nn.Linear(50,1))
projector = PCAManifoldProjector(problem.get_inputs(model).cpu())
optimizer = GeometricPhysicsGD(model.parameters(), lr=1e-3, rho=0.1, alpha=1.0)

def physics_residual():
    return problem.residual(model)
def noise_estimate():
    return local_entropy_1d(model, problem.get_inputs(model))

losses = []
for epoch in range(1000):
    def closure():
        optimizer.zero_grad()
        loss = problem.residual(model)
        loss.backward()
        return loss
    loss = optimizer.step(closure, projector, physics_residual, noise_estimate, problem.n_colloc, lambda: 0.1)
    losses.append(loss.item())
    if epoch % 200 == 0:
        print(f"Epoch {epoch}, loss = {loss.item():.4e}")

plt.semilogy(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('GPAGD on Poisson 1D'); plt.grid(True)
plt.show()